<a href="https://colab.research.google.com/github/bangaru01/C_programing/blob/main/XYZ_or_LOG_to_GJF_(Gaussian_input_file).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## XYZ-TO-GJF (TS)

In [ ]:
#!/usr/bin/env python3

import sys
from pathlib import Path

# ============================================================
# Gaussian settings
# ============================================================

# After adding xyz2gjf.py (this code)  to the Linux/server folder:
# Then: chmod +x xyz2gjf.py
# Then, in the folder containing all .xyz files:
#  "" for i in *.xyz; do ./xyz2gjf.py "$i"; done ""
# ============================================================
# ============================================================

MEMORY = "50GB"
NPROC = 64

ROUTE = (
    "# opt=(ts,calcfc,noeigen) freq=noraman "
    "6-31g(d,p) scrf=(solvent=dichloromethane) nosymm m062x"
)

CHARGE = 0
MULTIPLICITY = 1


def xyz_to_gjf(xyz_file):
    xyz_path = Path(xyz_file)

    if not xyz_path.exists():
        print(f"ERROR: File not found: {xyz_file}")
        return False

    if xyz_path.suffix.lower() != ".xyz":
        print(f"ERROR: Input file must have .xyz extension: {xyz_file}")
        return False

    # Output filename: input.xyz -> input.gjf
    gjf_path = xyz_path.with_suffix(".gjf")

    # Read XYZ
    with open(xyz_path, "r") as f:
        lines = f.readlines()

    if len(lines) < 2:
        print(f"ERROR: Invalid XYZ file: {xyz_file}")
        return False

    # First line = number of atoms
    try:
        natoms = int(lines[0].strip())
    except ValueError:
        print(f"ERROR: First line is not a valid atom count: {xyz_file}")
        return False

    # Second line = XYZ comment/title
    xyz_comment = lines[1].strip()

    # Coordinates start from line 3
    coordinate_lines = lines[2:2 + natoms]

    if len(coordinate_lines) != natoms:
        print(
            f"ERROR: Expected {natoms} atoms, "
            f"but found {len(coordinate_lines)} coordinate lines: {xyz_file}"
        )
        return False

    # Validate coordinates
    coordinates = []

    for line_number, line in enumerate(coordinate_lines, start=3):
        parts = line.split()

        if len(parts) < 4:
            print(
                f"ERROR: Invalid coordinate line {line_number} "
                f"in {xyz_file}: {line.strip()}"
            )
            return False

        element = parts[0]
        x = parts[1]
        y = parts[2]
        z = parts[3]

        coordinates.append(f"{element:<3} {x:>16} {y:>16} {z:>16}")

    # Use filename as Gaussian title
    title = xyz_path.stem

    # ========================================================
    # Write Gaussian input
    # ========================================================
    with open(gjf_path, "w") as f:

        f.write(f"%mem={MEMORY}\n")
        f.write(f"%nprocshared={NPROC}\n")
        f.write("\n")

        f.write(f"{ROUTE}\n")
        f.write("\n")

        f.write(f"{title}\n")
        f.write("\n")

        f.write(f"{CHARGE} {MULTIPLICITY}\n")

        for coord in coordinates:
            f.write(coord + "\n")

        f.write("\n")

    print(f"Created: {gjf_path}")

    return True


def main():

    if len(sys.argv) != 2:
        print("Usage:")
        print("  xyz2gjf.py input.xyz")
        print("")
        print("Example:")
        print("  ~/xyz2gjf.py input.xyz")
        sys.exit(1)

    xyz_to_gjf(sys.argv[1])


if __name__ == "__main__":
    main()

## XYZ TO GJF (mod)

In [ ]:
#!/usr/bin/env python3

import sys
from pathlib import Path

# ============================================================
# Gaussian settings :
# ============================================================

# After adding xyz2gjf.py (this code)  to the Linux/server folder:
# Then: chmod +x xyz2gjf.py
# Then, in the folder containing all .xyz files:
#  "" for i in *.xyz; do ./xyz2gjf.py "$i"; done ""
# ============================================================

MEMORY = "50GB"
NPROC = 64

ROUTE = (
    "# opt=modredundant "
    "6-31g(d,p) scrf=(solvent=dichloromethane) nosymm m062x"
)

CHARGE = 0
MULTIPLICITY = 1


def xyz_to_gjf(xyz_file):

    xyz_path = Path(xyz_file)

    if not xyz_path.exists():
        print(f"ERROR: File not found: {xyz_file}")
        return False

    if xyz_path.suffix.lower() != ".xyz":
        print(f"ERROR: Input file must have .xyz extension: {xyz_file}")
        return False

    # Output filename: input.xyz -> input.gjf
    gjf_path = xyz_path.with_suffix(".gjf")

    # --------------------------------------------------------
    # Read XYZ file
    # --------------------------------------------------------

    with open(xyz_path, "r") as f:
        lines = f.readlines()

    if len(lines) < 2:
        print(f"ERROR: Invalid XYZ file: {xyz_file}")
        return False

    # First line = number of atoms
    try:
        natoms = int(lines[0].strip())
    except ValueError:
        print(f"ERROR: First line is not a valid atom count: {xyz_file}")
        return False

    # Second line = XYZ comment
    xyz_comment = lines[1].strip()

    # --------------------------------------------------------
    # Read coordinates
    # --------------------------------------------------------

    coordinate_lines = lines[2:2 + natoms]

    if len(coordinate_lines) != natoms:
        print(
            f"ERROR: Expected {natoms} atoms, "
            f"but found {len(coordinate_lines)} coordinate lines: "
            f"{xyz_file}"
        )
        return False

    coordinates = []

    for line_number, line in enumerate(coordinate_lines, start=3):

        parts = line.split()

        if len(parts) < 4:
            print(
                f"ERROR: Invalid coordinate line {line_number} "
                f"in {xyz_file}: {line.strip()}"
            )
            return False

        element = parts[0]
        x = parts[1]
        y = parts[2]
        z = parts[3]

        coordinates.append(
            f"{element:<3} {x:>16} {y:>16} {z:>16}"
        )

    # --------------------------------------------------------
    # Gaussian title
    # --------------------------------------------------------

    title = xyz_path.stem

    # --------------------------------------------------------
    # Write Gaussian input file
    # --------------------------------------------------------

    with open(gjf_path, "w") as f:

        f.write(f"%mem={MEMORY}\n")
        f.write(f"%nprocshared={NPROC}\n")
        f.write("\n")

        f.write(f"{ROUTE}\n")
        f.write("\n")

        f.write(f"{title}\n")
        f.write("\n")

        f.write(f"{CHARGE} {MULTIPLICITY}\n")

        for coord in coordinates:
            f.write(coord + "\n")

        # Blank line after coordinates
        f.write("\n")

        # Freeze distance between atoms 17 and 22
        f.write("17 22 F\n")

        # Final blank line
        f.write("\n")

    print(f"Created: {gjf_path}")

    return True


def main():

    if len(sys.argv) != 2:
        print("Usage:")
        print("  ./xyz2gjf.py input.xyz")
        sys.exit(1)

    xyz_to_gjf(sys.argv[1])


if __name__ == "__main__":
    main()

## LOG TO GJF

In [ ]:
#!/usr/bin/env python3

from pathlib import Path


# ============================================================
# Gaussian TS input settings
# ============================================================

MEMORY = "50GB"
NPROC = 64

ROUTE = (
    "# opt=(ts,calcfc,noeigen) freq=noraman "
    "6-31g(d,p) scrf=(solvent=dichloromethane) nosymm m062x"
)

CHARGE = 0
MULTIPLICITY = 1


# ============================================================
# Atomic number -> element
# ============================================================

ELEMENTS = [
    None,
    "H", "He",
    "Li", "Be", "B", "C", "N", "O", "F", "Ne",
    "Na", "Mg", "Al", "Si", "P", "S", "Cl", "Ar",
    "K", "Ca", "Sc", "Ti", "V", "Cr", "Mn", "Fe", "Co", "Ni",
    "Cu", "Zn", "Ga", "Ge", "As", "Se", "Br", "Kr",
    "Rb", "Sr", "Y", "Zr", "Nb", "Mo", "Tc", "Ru", "Rh", "Pd",
    "Ag", "Cd", "In", "Sn", "Sb", "Te", "I", "Xe",
    "Cs", "Ba", "La", "Ce", "Pr", "Nd", "Pm", "Sm", "Eu", "Gd",
    "Tb", "Dy", "Ho", "Er", "Tm", "Yb", "Lu",
    "Hf", "Ta", "W", "Re", "Os", "Ir", "Pt", "Au", "Hg",
    "Tl", "Pb", "Bi", "Po", "At", "Rn",
    "Fr", "Ra", "Ac", "Th", "Pa", "U", "Np", "Pu", "Am", "Cm",
    "Bk", "Cf", "Es", "Fm", "Md", "No", "Lr",
    "Rf", "Db", "Sg", "Bh", "Hs", "Mt", "Ds", "Rg", "Cn",
    "Nh", "Fl", "Mc", "Lv", "Ts", "Og"
]


# ============================================================
# Find all Gaussian orientation blocks
# ============================================================

def extract_orientation_blocks(lines, orientation_name):

    blocks = []

    for i, line in enumerate(lines):

        if orientation_name not in line:
            continue

        # Find the coordinate header/dashed line structure
        dashed_count = 0
        start = None

        for j in range(i + 1, min(i + 10, len(lines))):

            if "-----" in lines[j]:
                dashed_count += 1

                # Second dashed line = beginning of coordinates
                if dashed_count == 2:
                    start = j + 1
                    break

        if start is None:
            continue

        coords = []

        for k in range(start, len(lines)):

            line_k = lines[k].strip()

            # End of orientation block
            if line_k.startswith("-----"):
                break

            parts = line_k.split()

            # Expected:
            # Center AtomicNumber AtomicType X Y Z
            if len(parts) < 6:
                continue

            try:
                center = int(parts[0])
                atomic_number = int(parts[1])
                atomic_type = int(parts[2])

                x = float(parts[3])
                y = float(parts[4])
                z = float(parts[5])

            except ValueError:
                continue

            coords.append(
                (center, atomic_number, atomic_type, x, y, z)
            )

        if coords:
            blocks.append(coords)

    return blocks


# ============================================================
# Convert coordinates to XYZ
# ============================================================

def write_xyz(xyz_path, coords, title):

    with open(xyz_path, "w") as f:

        f.write(f"{len(coords)}\n")
        f.write(f"{title}\n")

        for center, atomic_number, atomic_type, x, y, z in coords:

            if atomic_number >= len(ELEMENTS):
                element = "X"
            else:
                element = ELEMENTS[atomic_number]

            f.write(
                f"{element:<3} "
                f"{x:16.8f} "
                f"{y:16.8f} "
                f"{z:16.8f}\n"
            )


# ============================================================
# Create Gaussian TS input
# ============================================================

def write_gjf(gjf_path, coords, title):

    with open(gjf_path, "w") as f:

        f.write(f"%mem={MEMORY}\n")
        f.write(f"%nprocshared={NPROC}\n")
        f.write("\n")

        f.write(f"{ROUTE}\n")
        f.write("\n")

        f.write(f"{title}\n")
        f.write("\n")

        f.write(f"{CHARGE} {MULTIPLICITY}\n")

        for center, atomic_number, atomic_type, x, y, z in coords:

            if atomic_number >= len(ELEMENTS):
                element = "X"
            else:
                element = ELEMENTS[atomic_number]

            f.write(
                f"{element:<3} "
                f"{x:16.8f} "
                f"{y:16.8f} "
                f"{z:16.8f}\n"
            )

        # IMPORTANT:
        # No "17 22 F" here.
        # The frozen 17-22 distance was only for Stage 1.
        f.write("\n")


# ============================================================
# Process one Gaussian log
# ============================================================

def process_log(log_file):

    print()
    print("=" * 70)
    print(f"Processing: {log_file.name}")

    with open(log_file, "r", errors="ignore") as f:
        lines = f.readlines()

    # --------------------------------------------------------
    # Check whether Gaussian optimization completed
    # --------------------------------------------------------

    optimization_completed = any(
        "Optimization completed." in line
        for line in lines
    )

    if not optimization_completed:

        print("WARNING: Optimization completed. not found.")
        print("Skipping this file.")
        return False

    # --------------------------------------------------------
    # First choice: Standard orientation
    # --------------------------------------------------------

    standard_blocks = extract_orientation_blocks(
        lines,
        "Standard orientation:"
    )

    # --------------------------------------------------------
    # Fallback: Input orientation
    # --------------------------------------------------------

    input_blocks = extract_orientation_blocks(
        lines,
        "Input orientation:"
    )

    if standard_blocks:

        coords = standard_blocks[-1]
        orientation_used = "last Standard orientation"

    elif input_blocks:

        coords = input_blocks[-1]
        orientation_used = "last Input orientation"

    else:

        print("ERROR: No Gaussian orientation block found.")
        return False

    # --------------------------------------------------------
    # Check number of atoms
    # --------------------------------------------------------

    natoms = len(coords)

    if natoms == 0:

        print("ERROR: No coordinates extracted.")
        return False

    print(f"Optimization completed: YES")
    print(f"Geometry used: {orientation_used}")
    print(f"Number of atoms: {natoms}")

    # --------------------------------------------------------
    # Output filenames
    # --------------------------------------------------------

    stem = log_file.stem

    xyz_file = log_file.parent / f"{stem}_final.xyz"
    gjf_file = log_file.parent / f"{stem}_TS.gjf"

    # --------------------------------------------------------
    # Write XYZ
    # --------------------------------------------------------

    write_xyz(
        xyz_file,
        coords,
        f"{stem} final optimized geometry"
    )

    print(f"Created XYZ: {xyz_file.name}")

    # --------------------------------------------------------
    # Write TS GJF
    # --------------------------------------------------------

    write_gjf(
        gjf_file,
        coords,
        f"{stem} TS optimization"
    )

    print(f"Created GJF: {gjf_file.name}")

    return True


# ============================================================
# Main
# ============================================================

def main():

    current_dir = Path.cwd()

    log_files = sorted(current_dir.glob("*.log"))

    if not log_files:

        print("No .log files found in the current directory.")
        return

    print("=" * 70)
    print("Gaussian LOG -> FINAL XYZ -> TS GJF")
    print("=" * 70)
    print(f"Directory: {current_dir}")
    print(f"LOG files found: {len(log_files)}")

    successful = 0
    skipped = 0

    for log_file in log_files:

        result = process_log(log_file)

        if result:
            successful += 1
        else:
            skipped += 1

    print()
    print("=" * 70)
    print("SUMMARY")
    print("=" * 70)
    print(f"LOG files found : {len(log_files)}")
    print(f"Processed       : {successful}")
    print(f"Skipped         : {skipped}")
    print("=" * 70)


if __name__ == "__main__":
    main()